# Stage 11C: nested delta-U weight/cap and absolute-tail gate

This standalone Colab notebook reuses the Stage 11 coefficient OOF artifacts. It selects correction weight and cap without seeing each outer fold, replaces the overly strict tail-share gate with absolute worst-tail SSE/CVaR/P90 checks, and confirms one inference profile across ordinary, spatial, and typewell holdouts. It does not create a Kaggle submission.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Reuse Stage 11

The completed 773-well Stage 11 run is reused. If it is absent, the cell rebuilds it once. Keep the same Google Drive data layout used by notebook 250.


In [ ]:
STAGE11_RUN_ID = 'stage11_multicut_delta_u_full_v001'
stage11_run = artifact_dir / STAGE11_RUN_ID
if not (stage11_run / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-delta-u',
        '--config', 'configs/experiment/stage11_multicut_delta_u.yaml',
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', STAGE11_RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing Stage 11:', stage11_run)


## Run robust nested gate

This is a CPU/high-RAM evaluation. Twelve predeclared weight/cap profiles are reconstructed for all three holdout families. No model is chosen using its own outer fold.


In [ ]:
RUN_ID = 'stage11c_delta_u_robust_gate_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-delta-u-gate',
        '--config', 'configs/experiment/stage11c_delta_u_robust_gate.yaml',
        '--stage11-run', str(stage11_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ]
    log_path = artifact_dir / f'{RUN_ID}_driver.log'
    with log_path.open('w', encoding='utf-8') as log_handle:
        process = subprocess.Popen(
            command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_handle.write(line); log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        print('\n'.join(tail))
        raise RuntimeError(f'Stage 11C failed with exit code {return_code}. Full log: {log_path}')
else:
    print('Reusing completed run:', run_dir)


## Decision summary

`promoted_to_stage12: True` authorizes the independent GR-alignment benchmark. It still does not authorize a Kaggle submission.


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text(encoding='utf-8'))
{
    'promoted_to_stage12': summary['promoted_to_stage12'],
    'base_rmse': summary['standard_base_rmse'],
    'nested_rmse': summary['standard_nested_rmse'],
    'rmse_delta': summary['standard_nested_delta'],
    'spatial_delta': summary['spatial_nested_delta'],
    'typewell_delta': summary['typewell_nested_delta'],
    'bootstrap_95pct': [summary['bootstrap']['ci_2_5'], summary['bootstrap']['ci_97_5']],
    'improved_folds': f"{summary['improved_standard_folds']}/{summary['required_standard_folds']}",
    'selected_inference_spec': summary['selected_inference_spec'],
    'selected_inference_parameters': summary['selected_inference_parameters'],
    'worst_tail_share_diagnostic_delta': summary['worst_tail_share_diagnostic_delta'],
    'gates': summary['gates'],
    'standard_selections': summary['nested_selections']['fold'],
    'cut_report': summary['cut_report'],
    'next_step': summary['next_step'],
}


In [ ]:
import pandas as pd
spec_report = pd.read_parquet(run_dir / 'spec_report.parquet')
spec_report.sort_values(['family', 'pooled_rmse']).groupby('family').head(6).reset_index(drop=True)


## Send back

Send the full decision dictionary and the displayed top-spec table. The selections reveal whether one strong profile generalizes or the gain depends on fold-specific tuning.
